In [ ]:
import h5py
import torch

from src.utils.data_utils import load_metadata
from src.models.components.sph import relax_wrapper
from src.utils.nbrs_utils import shift_fn

## 1. Check for correct chunking

In [ ]:
# show the shapes of the leave objects in the h5 dataset
path = "/local/disk/atoshev/dataset_kolm/datasets/2D_KOLM_4096/valid.h5"


def show_shapes(h5file):
    for key in h5file.keys():
        print(f"=== {key} ===")
        data = h5file[key]
        if isinstance(data, h5py.Dataset):
            print(f"Shape: {data.shape}, Dtype: {data.dtype}")
            if data.chunks is not None:
                num_chunks = tuple((s + c - 1) // c for s, c in zip(data.shape, data.chunks))
                print(f"There are {num_chunks} chunks of shape {data.chunks}")
            else:
                print("No chunks")
        elif isinstance(data, h5py.Group):
            print("This is a group, not a dataset.")
        else:
            print("Unknown type.")


with h5py.File(path, "r") as f:
    traj_0 = f["00000"]
    show_shapes(traj_0)

## 2. Check for correct dataset evolution <-> u2v

In [ ]:
# load trajectory from dataset

data_root = "../data/2D_KOLM_4096"
split = "train"
steps = 100
with h5py.File(f"{data_root}/{split}.h5", "r") as f:
    r = torch.tensor(f["00000/position"][:steps])
    u_r = torch.tensor(f["00000/u"][:steps])

print(r.shape, u_r.shape)
print(u_r[:, 0, 0])  # velocty in x of particle 0 over time

In [ ]:
# Based on: scr/models/gino_module.py

# take the first frame of the particle system and reproduce the datase generation process by
# using the ground truth u field

metadata = load_metadata(data_root)
boundaries = torch.tensor(metadata["bounds"], requires_grad=False).float()
boundaries = boundaries[:, 1] - boundaries[:, 0]
pbc = metadata["periodic_boundary_conditions"]
# dt = metadata["dt"]
_effective_dt = metadata["dt"] * metadata["write_every"]

rlx_fn = relax_wrapper(
    Nx=int(round(r[0].shape[0]) ** (1 / metadata["dim"])),
    dim=metadata["dim"],
    L=boundaries[0].item(),
    is_physical=True,
    u_ref=metadata["u_ref"],
    is_tvf=metadata["rlx_is_tvf"],
    nu=0.0,
    box=boundaries,
)


def _sph_rlx(r, n_part_per_traj, dt_factor, num_steps):
    v = 0.0
    for i in range(num_steps):
        a_temp = rlx_fn(r, n_part_per_traj)
        dr = (dt_factor * metadata["dt"]) ** 2 * a_temp
        r = shift_fn(r, dr)
        v += dr
    return v, v, r


def _u2v(u):
    return u * _effective_dt


x = r[0].clone()
traj = [x.clone()]
n_particles_per_trajectory = torch.tensor([x.shape[0]], dtype=torch.int64)
for i in range(steps - 1):
    u_i = u_r[i]
    x = shift_fn(x, _u2v(u_i), boundaries, pbc)
    _, _, x = _sph_rlx(
        x,
        n_particles_per_trajectory,
        dt_factor=metadata["rlx_dt_factor"],
        num_steps=metadata["rlx_num_steps"],
    )
    traj.append(x.clone())
traj = torch.stack(traj, dim=0)

In [ ]:
# difference between the frames after 100 steps:
traj[-5:, 0, 0], r[-5:, 0, 0]